# Traditional Text Analysis

This notebook focuses on non-LLM based analysis of reasoning, including length, vocabulary diversity, and factor detection.

## Overview
- Load model reasoning data
- Compute basic metrics (word count, sentence count)
- Analyze vocabulary diversity
- Detect mentions of key factors (e.g., 'infection rate', 'economy')
- Analyze context awareness via keyword matching
- Assessment of logical structure markers
- Generate summary statistics
- Save detailed results for visualization

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import json
import sys

# Add current directory to path
sys.path.append(os.getcwd())

from config import *
from utils import load_all_data, save_analysis_results

import re
from collections import Counter

# Set random seeds
np.random.seed(RANDOM_SEED)

print("✓ All imports successful and config loaded")

In [ ]:
# Configuration imported from config.py

print(f"SUBSAMPLE_SIZE: {SUBSAMPLE_SIZE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


In [ ]:
class ReasoningAnalyzer:
    """Traditional text-based analysis of reasoning"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        
    def compute_basic_metrics(self) -> pd.DataFrame:
        """Calculate basic quantitative metrics for reasoning text"""
        
        # Length metrics
        self.df['reasoning_length'] = self.df['reasoning'].str.len()
        self.df['reasoning_word_count'] = self.df['reasoning'].str.split().str.len()
        self.df['reasoning_sentence_count'] = self.df['reasoning'].str.count(r'[.!?]+')
        self.df['avg_word_length'] = self.df['reasoning'].apply(
            lambda x: np.mean([len(word) for word in str(x).split()]) if pd.notna(x) and len(str(x).split()) > 0 else 0
        )
        self.df['avg_sentence_length'] = (
            self.df['reasoning_word_count'] / self.df['reasoning_sentence_count'].replace(0, 1)
        )
        
        return self.df
    
    def compute_vocabulary_diversity(self) -> pd.DataFrame:
        """Measure lexical diversity and complexity"""
        
        def type_token_ratio(text):
            """Unique words / total words"""
            if pd.isna(text) or len(str(text).split()) == 0:
                return 0
            words = str(text).lower().split()
            return len(set(words)) / len(words)
        
        self.df['vocabulary_diversity'] = self.df['reasoning'].apply(type_token_ratio)
        
        return self.df
    
    def detect_factor_mentions(self) -> pd.DataFrame:
        """Detect mentions of key decision factors"""
        
        # Define factor keywords
        factor_keywords = {
            'infection_rate': r'\b(infection|case|spread|disease|transmission|epidemic)\b',
            'npi_measures': r'\b(restriction|lockdown|mandate|measure|policy|intervention|npi)\b',
            'economic': r'\b(economic|economy|financial|business|cost|employment|job)\b',
            'capacity': r'\b(capacity|hospital|healthcare|resource|bed|facility)\b',
            'social': r'\b(social|community|public|people|population|society)\b',
            'comparative': r'\b(compar|versus|vs|between|option|alternative|choice)\b',
            'quantitative': r'\b(\d+%|\d+\.\d+|higher|lower|more|less|reduce|increase)\b',
            'causal': r'\b(because|therefore|thus|since|due to|as a result|leads to|causes)\b'
        }
        
        for factor, pattern in factor_keywords.items():
            self.df[f'mentions_{factor}'] = self.df['reasoning'].str.contains(
                pattern, case=False, na=False, regex=True
            ).astype(int)
        
        # Total factor coverage
        mention_cols = [col for col in self.df.columns if col.startswith('mentions_')]
        self.df['total_factors_mentioned'] = self.df[mention_cols].sum(axis=1)
        
        return self.df
    
    def detect_context_awareness(self) -> pd.DataFrame:
        """Check if context variables are explicitly mentioned"""
        
        # These will vary based on your strategies
        context_patterns = {
            'capacity_mentioned': r'\b(capacity|hospital|healthcare system)\b',
            'economic_mentioned': r'\b(economic|economy|financial impact)\b',
            'preference_mentioned': r'\b(prefer|preference|priority)\b'
        }
        
        for var, pattern in context_patterns.items():
            self.df[f'context_{var}'] = self.df['reasoning'].str.contains(
                pattern, case=False, na=False, regex=True
            ).astype(int)
        
        return self.df
    
    def compute_logical_structure_metrics(self) -> pd.DataFrame:
        """Analyze logical structure of reasoning"""
        
        # Count logical connectors
        self.df['has_comparison'] = self.df['reasoning'].str.contains(
            r'\b(however|whereas|while|although|but|compare|versus)\b',
            case=False, na=False, regex=True
        ).astype(int)
        
        self.df['has_causation'] = self.df['reasoning'].str.contains(
            r'\b(because|therefore|thus|hence|consequently|as a result)\b',
            case=False, na=False, regex=True
        ).astype(int)
        
        self.df['has_conclusion'] = self.df['reasoning'].str.contains(
            r'\b(therefore|thus|consequently|in conclusion|overall)\b',
            case=False, na=False, regex=True
        ).astype(int)
        
        return self.df
    
    def analyze_consistency_within_model(self) -> pd.DataFrame:
        """Calculate within-model consistency metrics"""
        
        # Group by model and similar contexts
        consistency_metrics = []
        
        for model in self.df['model'].unique():
            model_df = self.df[self.df['model'] == model]
            
            # Calculate variability in reasoning length
            length_mean = model_df['reasoning_word_count'].mean()
            length_cv = model_df['reasoning_word_count'].std() / length_mean if length_mean > 0 else 0
            
            # Calculate variability in factor mentions
            factor_cols = [col for col in self.df.columns if col.startswith('mentions_')]
            factor_consistency = model_df[factor_cols].std().mean()
            
            consistency_metrics.append({
                'model': model,
                'reasoning_length_cv': length_cv,
                'factor_mention_std': factor_consistency,
                'n_samples': len(model_df)
            })
        
        return pd.DataFrame(consistency_metrics)
    
    def generate_summary_statistics(self) -> pd.DataFrame:
        """Generate summary statistics by model and context"""
        
        # Group by model
        summary = self.df.groupby('model').agg({
            'reasoning_word_count': ['mean', 'std', 'min', 'max'],
            'vocabulary_diversity': ['mean', 'std'],
            'total_factors_mentioned': ['mean', 'std'],
            'has_comparison': 'mean',
            'has_causation': 'mean',
            'has_conclusion': 'mean'
        }).round(3)
        
        return summary

In [ ]:
# Load data using utility function
results_df = load_all_data(STRATEGIES, subsample_size=SUBSAMPLE_SIZE, random_seed=RANDOM_SEED)


In [ ]:
# Run Traditional Analysis
if not results_df.empty:
    analyzer = ReasoningAnalyzer(results_df)
    
    print("Running metrics calculation...")
    results_df = analyzer.compute_basic_metrics()
    results_df = analyzer.compute_vocabulary_diversity()
    results_df = analyzer.detect_factor_mentions()
    results_df = analyzer.detect_context_awareness()
    results_df = analyzer.compute_logical_structure_metrics()
    
    # Generate and display summary
    print("\nTraditional Metrics Summary:")
    summary = analyzer.generate_summary_statistics()
    display(summary)
    
    print("\nModel Consistency Analysis:")
    consistency = analyzer.analyze_consistency_within_model()
    display(consistency)
    
    # Save Results Granularly
    save_analysis_results(results_df, TRADITIONAL_DIR, 'traditional')
